<a href="https://colab.research.google.com/github/Anshulmohan27/Python_Everything/blob/main/Chess_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import random #importing random class to work on random move selection for possible moves in the list

chess_board = []

black_starting_pieces = [
    ['r', 'n', 'b', 'q', 'k', 'b', 'n', 'r'],
    ['p', 'p', 'p', 'p', 'p', 'p', 'p', 'p'],
]
white_starting_pieces = [
    ['P', 'P', 'P', 'P', 'P', 'P', 'P', 'P'],
    ['R', 'N', 'B', 'Q', 'K', 'B', 'N', 'R'],
]

chess_board.extend(black_starting_pieces)
for _ in range(4):
    chess_board.append(['.'] * 8)
chess_board.extend(white_starting_pieces)

white_king_position = [7, 4]
black_king_position = [0, 4]

def print_board(board):
    for i, row in enumerate(board):
        print(' '.join(row), 8 - i)
    print('a b c d e f g h')

def is_own_piece(cell, color):
    if cell == '.':
        return False
    return cell.isupper() if color == 'white' else cell.islower()

def is_opponent(cell, color):
    if cell == '.':
        return False
    return cell.islower() if color == 'white' else cell.isupper()

def check_move_within_board(from_col, from_row, to_col, to_row):
    return (from_col in range(97, 105) and from_row in range(1, 9) and
            to_col   in range(97, 105) and to_row   in range(1, 9) and
            (from_col, from_row) != (to_col, to_row))

def check_if_vertical_move_is_possible(from_col, from_row, to_col, to_row, color):
    step = 1 if to_row > from_row else -1
    for i in range(from_row + step, to_row + step, step):
        cell = chess_board[8 - i][from_col - 97]
        if i != to_row:
            if cell != '.':
                return False
        else:
            if is_own_piece(cell, color):
                return False
    return True

def check_if_horizontal_move_is_possible(from_col, from_row, to_col, to_row, color):
    step = 1 if to_col > from_col else -1
    for i in range(from_col + step, to_col + step, step):
        cell = chess_board[8 - from_row][i - 97]
        if i != to_col:
            if cell != '.':
                return False
        else:
            if is_own_piece(cell, color):
                return False
    return True

def check_if_diagonal_move_is_possible(from_col, from_row, to_col, to_row, color):
    if abs(to_row - from_row) != abs(to_col - from_col):
        return False
    row_step = 1 if to_row > from_row else -1
    col_step = 1 if to_col > from_col else -1
    steps = abs(to_row - from_row)
    for i in range(1, steps + 1):
        cell = chess_board[8 - (from_row + i * row_step)][from_col + i * col_step - 97]
        if i < steps:
            if cell != '.':
                return False
        else:
            if is_own_piece(cell, color):
                return False
    return True

def Pawn_permissible_moves(from_col, from_row, to_col, to_row, color):
    direction = 1 if color == 'white' else -1
    start_row = 2 if color == 'white' else 7
    row_diff = to_row - from_row
    if row_diff == direction:
        if to_col == from_col:
            return chess_board[8 - to_row][to_col - 97] == '.'
        if abs(to_col - from_col) == 1:
            return is_opponent(chess_board[8 - to_row][to_col - 97], color)
    if row_diff == 2 * direction and from_row == start_row:
        if to_col == from_col:
            mid = chess_board[8 - (from_row + direction)][from_col - 97]
            dest = chess_board[8 - to_row][to_col - 97]
            return mid == '.' and dest == '.'
    return False

def Rook_permissible_moves(from_col, from_row, to_col, to_row, color):
    if to_col == from_col:
        return check_if_vertical_move_is_possible(from_col, from_row, to_col, to_row, color)
    if to_row == from_row:
        return check_if_horizontal_move_is_possible(from_col, from_row, to_col, to_row, color)
    return False

def Bishop_permissible_moves(from_col, from_row, to_col, to_row, color):
    return check_if_diagonal_move_is_possible(from_col, from_row, to_col, to_row, color)

def Knight_permissible_moves(from_col, from_row, to_col, to_row, color):
    row_diff = abs(to_row - from_row)
    col_diff = abs(to_col - from_col)
    if not ((row_diff == 2 and col_diff == 1) or (row_diff == 1 and col_diff == 2)):
        return False
    dest = chess_board[8 - to_row][to_col - 97]
    return not is_own_piece(dest, color)

def King_permissible_moves(from_col, from_row, to_col, to_row, color):
    if abs(to_row - from_row) > 1 or abs(to_col - from_col) > 1:
        return False
    if to_col == from_col:
        return check_if_vertical_move_is_possible(from_col, from_row, to_col, to_row, color)
    if to_row == from_row:
        return check_if_horizontal_move_is_possible(from_col, from_row, to_col, to_row, color)
    return check_if_diagonal_move_is_possible(from_col, from_row, to_col, to_row, color)

def Queen_permissible_moves(from_col, from_row, to_col, to_row, color):
    return (Rook_permissible_moves(from_col, from_row, to_col, to_row, color) or
            Bishop_permissible_moves(from_col, from_row, to_col, to_row, color))

def is_in_check(color):
    king_piece = 'K' if color == 'white' else 'k'
    king_pos = None
    for row_idx in range(8):
        for col_idx in range(8):
            if chess_board[row_idx][col_idx] == king_piece:
                king_pos = (col_idx + 97, 8 - row_idx)
                break
        if king_pos:
            break
    if not king_pos:
        return False
    king_col, king_row = king_pos
    opponent = 'black' if color == 'white' else 'white'
    for row_idx in range(8):
        for col_idx in range(8):
            cell = chess_board[row_idx][col_idx]
            if cell == '.' or not is_own_piece(cell, opponent):
                continue
            from_row = 8 - row_idx
            from_col = col_idx + 97
            if check_if_legal_move(from_col, from_row, king_col, king_row, opponent):
                return True
    return False

def generate_all_legal_moves(color):
    legal_moves = []
    for row_idx in range(8):
        for col_idx in range(8):
            cell = chess_board[row_idx][col_idx]
            if cell == '.' or not is_own_piece(cell, color):
                continue
            from_row = 8 - row_idx
            from_col = col_idx + 97
            for to_row in range(1, 9):
                for to_col in range(97, 105):
                    if not check_if_legal_move(from_col, from_row, to_col, to_row, color):
                        continue
                    captured = chess_board[8 - to_row][to_col - 97]
                    chess_board[8 - to_row][to_col - 97] = chess_board[8 - from_row][from_col - 97]
                    chess_board[8 - from_row][from_col - 97] = '.'
                    in_check = is_in_check(color)
                    chess_board[8 - from_row][from_col - 97] = chess_board[8 - to_row][to_col - 97]
                    chess_board[8 - to_row][to_col - 97] = captured
                    if not in_check:
                        legal_moves.append((from_col, from_row, to_col, to_row))
    return legal_moves

def board_state(board, white_moves, black_moves):
    # white_moves and black_moves passed in — avoids 2 extra generate_all_legal_moves calls
    piece_value = {
        'P': 100, 'R': 500, 'N': 300, 'B': 300, 'Q': 900, 'K': 0,
        'p':-100, 'r':-500, 'n':-300, 'b':-300, 'q':-900, 'k': 0
    }
    mobility_score = white_moves - black_moves
    total = 0
    for row in board:
        for piece in row:
            if piece in piece_value:
                total += piece_value[piece]
    return total + mobility_score

PIECE_VALIDATORS = {
    'P': Pawn_permissible_moves, 'p': Pawn_permissible_moves,
    'R': Rook_permissible_moves, 'r': Rook_permissible_moves,
    'B': Bishop_permissible_moves, 'b': Bishop_permissible_moves,
    'N': Knight_permissible_moves, 'n': Knight_permissible_moves,
    'K': King_permissible_moves, 'k': King_permissible_moves,
    'Q': Queen_permissible_moves, 'q': Queen_permissible_moves,
}

def check_if_legal_move(from_col, from_row, to_col, to_row, color):
    global white_king_position, black_king_position
    if not check_move_within_board(from_col, from_row, to_col, to_row):
        return False
    piece = chess_board[8 - from_row][from_col - 97]
    if not is_own_piece(piece, color):
        return False
    validator = PIECE_VALIDATORS.get(piece)
    if not validator:
        return False
    result = validator(from_col, from_row, to_col, to_row, color)
    if result and piece in ('K', 'k'):
        if color == 'white':
            white_king_position = [8 - to_row, to_col - 97]
        else:
            black_king_position = [8 - to_row, to_col - 97]
    return result

def is_checkmate(color, legal_moves):
    # legal_moves passed in — avoids calling generate_all_legal_moves a second time
    return is_in_check(color) and len(legal_moves) == 0

def is_stalemate(color, legal_moves):
    # legal_moves passed in — avoids calling generate_all_legal_moves a second time
    return not is_in_check(color) and len(legal_moves) == 0

def black_random_move():
    moves = generate_all_legal_moves('black')
    if not moves:
        if is_in_check('black'):
            print("Checkmate! White wins!")
        else:
            print("Stalemate! It's a draw!")
        return False
    # Compute white move count once — reused for every board_state call in the loop
    white_move_count = len(generate_all_legal_moves('white'))
    best_move = None
    best_score = float('inf')
    for move in moves:
        from_col, from_row, to_col, to_row = move
        captured = chess_board[8 - to_row][to_col - 97]
        chess_board[8 - to_row][to_col - 97] = chess_board[8 - from_row][from_col - 97]
        chess_board[8 - from_row][from_col - 97] = '.'
        # Pass precomputed counts — board_state no longer calls generate_all_legal_moves
        score = board_state(chess_board, white_move_count, len(moves))
        chess_board[8 - from_row][from_col - 97] = chess_board[8 - to_row][to_col - 97]
        chess_board[8 - to_row][to_col - 97] = captured
        if score < best_score:
            best_score = score
            best_move = move
    from_col, from_row, to_col, to_row = best_move
    chess_board[8 - to_row][to_col - 97]     = chess_board[8 - from_row][from_col - 97]
    chess_board[8 - from_row][from_col - 97] = '.'
    print(f"Black plays: {chr(from_col)}{from_row} → {chr(to_col)}{to_row}")
    return True

print_board(chess_board)

while True:

    move = input("Your move (e.g. e2 e4): ").strip()
    if move == 'quit':
        break

    _from, _to = move.split()
    from_col, from_row = ord(_from[0]), int(_from[1])
    to_col,   to_row   = ord(_to[0]),   int(_to[1])

    if not check_if_legal_move(from_col, from_row, to_col, to_row, 'white'):
        print("Illegal move — try again.")
        continue

    chess_board[8 - to_row][to_col - 97]     = chess_board[8 - from_row][from_col - 97]
    chess_board[8 - from_row][from_col - 97] = '.'

    # Generate black's legal moves ONCE — reuse for both checkmate and stalemate checks
    black_legal = generate_all_legal_moves('black')
    if is_checkmate('black', black_legal):
        print_board(chess_board)
        print("Checkmate! White wins!")
        break
    if is_stalemate('black', black_legal):
        print_board(chess_board)
        print("Stalemate! It's a draw!")
        break

    print_board(chess_board)

    if not black_random_move():
        print_board(chess_board)
        break

    # Generate white's legal moves ONCE — reuse for both checkmate and stalemate checks
    white_legal = generate_all_legal_moves('white')
    if is_checkmate('white', white_legal):
        print_board(chess_board)
        print("Checkmate! Black wins!")
        break
    if is_stalemate('white', white_legal):
        print_board(chess_board)
        print("Stalemate! It's a draw!")
        break

    print_board(chess_board)

r n b q k b n r 8
p p p p p p p p 7
. . . . . . . . 6
. . . . . . . . 5
. . . . . . . . 4
. . . . . . . . 3
P P P P P P P P 2
R N B Q K B N R 1
a b c d e f g h
Your move (e.g. e2 e4): e2 e4
r n b q k b n r 8
p p p p p p p p 7
. . . . . . . . 6
. . . . . . . . 5
. . . . P . . . 4
. . . . . . . . 3
P P P P . P P P 2
R N B Q K B N R 1
a b c d e f g h
Black plays: b8 → a6
r . b q k b n r 8
p p p p p p p p 7
n . . . . . . . 6
. . . . . . . . 5
. . . . P . . . 4
. . . . . . . . 3
P P P P . P P P 2
R N B Q K B N R 1
a b c d e f g h
Your move (e.g. e2 e4): quit
